In [1]:
import sys
from pathlib import Path

In [2]:
# in jupyter (lab / notebook), based on notebook path
module_path = str(Path.cwd().parents[0])

# # in standard python
# module_path = str(Path.cwd(__file__).parents[0] / "py")

if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import wandb
import torch

import data.donut_dataset as donut_dataset
import utils.helpers as helpers
import utils.push_to_hub as push_to_hub
import pytorch_lightning as pl

from data.donut_dataset import DonutDataset
from models.donut_pytorch_lightning import DonutModelPLModule
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import EarlyStopping

/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
image_path = "/leonardo_work/IscrC_HeR-T/weiwei/data/20240517/HeR-T_data/img_640x480"
max_length = 768

### Run when loading new data

In [11]:
dataset = donut_dataset.data_loader(image_path)

print('Data Loading completes.')

this is the dataset DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 32165
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 6893
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 6893
    })
})
Data Loading completes.


In [6]:
# torch.save(dataset, '/leonardo_work/IscrC_HeR-T/weiwei/data/20240517/HeR-T_data/img_1280x960/dataloader.pth')

### Run when loading a saved data loader

In [5]:
# dataset = torch.load('/leonardo_work/IscrC_HeR-T/weiwei/data/20240517/HeR-T_data/img_1280x960/dataloader.pth')

### Load the base processor and model

In [12]:
processor, model = donut_dataset.model_loader(dataset, max_length, "/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/models/donut-base")

print('Processor and Model are loaded.')

Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration.


Processor and Model are loaded.


### Run when customizing new datasets

In [13]:
train_dataset = DonutDataset(image_path, max_length=max_length,
                             split="train", task_start_token="<s_herbarium>", 
                             prompt_end_token="<s_herbarium>",
                             sort_json_key=False, # cord dataset is preprocessed -> no need for this
                             model=model, 
                             processor=processor,
                             )

print("Donut Training Dataset is loaded.")

val_dataset = DonutDataset(image_path, max_length=max_length,
                             split="validation", task_start_token="<s_herbarium>", 
                             prompt_end_token="<s_herbarium>",
                             sort_json_key=False, # cord dataset is preprocessed -> no need for this
                             model=model,
                             processor=processor,
                             )

print("Donut Validation Dataset is loaded.")

/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/PIL/TiffImagePlugin.py:870: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Donut Training Dataset is loaded.
Donut Validation Dataset is loaded.


In [14]:
# the vocab size attribute stays constants (might be a bit unintuitive - but doesn't include special tokens)
print("Original number of tokens:", processor.tokenizer.vocab_size)
print("Number of tokens after adding special tokens:", len(processor.tokenizer))

Original number of tokens: 57522
Number of tokens after adding special tokens: 57538


In [15]:
processor.decode([57537])

'<s_herbarium>'

In [10]:
# torch.save(train_dataset, '/leonardo_work/IscrC_HeR-T/weiwei/data/20240517/HeR-T_data/img_1280x960/train.pt')
# torch.save(val_dataset, '/leonardo_work/IscrC_HeR-T/weiwei/data/20240517/HeR-T_data/img_1280x960/val.pt')

### Run when using saved customized pytorch datasets

In [8]:
# train_dataset = torch.load('/leonardo_work/IscrC_HeR-T/weiwei/data/20240517/HeR-T_data/img_1280x960/train.pt')
# val_dataset = torch.load('/leonardo_work/IscrC_HeR-T/weiwei/data/20240517/HeR-T_data/img_1280x960/val.pt')

### Set up configurations of processor and model

In [16]:
processor.image_processor.size = helpers.image_size(dataset)
processor.image_processor.do_align_long_axis = False

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s_herbarium>'])[0]
model.config.hidden_dropout_prob = 0.2
model.config.attention_probs_dropout_prob = 0.2

print("Model and processor settings are completed.")

Model and processor settings are completed.


In [17]:
# sanity check
print("Pad token ID:", processor.decode([model.config.pad_token_id]))
print("Decoder start token ID:", processor.decode([model.config.decoder_start_token_id]))

Pad token ID: <pad>
Decoder start token ID: <s_herbarium>


In [18]:
config = {
    'max_epochs': 6,
    'val_check_interval': 0.25,
    'check_val_every_n_epoch': 1,
    'gradient_clip_val': 1.0,
    'num_training_samples_per_epoch': 32165,
    'lr': 2.5e-5, # or 2e-5
    'weight_decay': 2e-5,
    'dropout_rate': 0.2,
    'train_batch_sizes': 16,
    'val_batch_sizes': 16,
    'num_nodes': 1,
    'warmup_steps': 2500,
    'result_path': "/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/results",
    'verbose': True, 
    'seed': 16, 
    'num_workers' : 4
}

In [19]:
model_lightning = DonutModelPLModule(config, processor, model, train_dataset, val_dataset)

print('PyTorch Lightning Model has been set up.')

PyTorch Lightning Model has been set up.


In [18]:
# # wandb for online mode
# # api key so that it doesn't ask me for it
# wandb.login(key=confidential.api_key)
# wandb_logger = WandbLogger(project="HeR-T-trial", name="localTrial")

# # use default patiente
# early_stop_callback = EarlyStopping(monitor="val_edit_distance", verbose=True, mode="min") 

wandb: Network error (ConnectionError), entering retry loop.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /leonardo/home/userexternal/wliu0000/.netrc
wandb: Network error (ConnectionError), entering retry loop.


In [20]:
# wandb offline mode specially for CINECA Leonardo booster
wandb.init(mode="offline")
wandb_logger = WandbLogger(project="HeR-T-trial-241114", name="480x640_batchsize16")

# use default patiente
early_stop_callback = EarlyStopping(monitor="val_edit_distance", verbose=True, mode="min") 

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


In [21]:
pushToHub = push_to_hub.PushToHubCallback()

In [22]:
trainer = pl.Trainer(
        # accelerator="mps",  # only for local test on Macbook with silicone chips
        accelerator="gpu",
        devices=4,
        max_epochs=config['max_epochs'],
        val_check_interval=config['val_check_interval'],
        check_val_every_n_epoch=config['check_val_every_n_epoch'],
        gradient_clip_val=config['gradient_clip_val'],
        precision='bf16-mixed', # we'll use mixed precision
        num_sanity_val_steps=0,
        logger=wandb_logger,
        callbacks=[pushToHub, early_stop_callback]
)

/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/p ...
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(model_lightning)

In [ ]:
wandb.finish()